In [1]:
import os
import json
import boto3
from botocore import UNSIGNED
from botocore.config import Config

connexion AWS S3

In [2]:
# Public S3 client
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

# SpaceNet bucket and folders
BUCKET_NAME = 'spacenet-dataset'
BASE_PREFIX = 'spacenet/SN8_floods/Germany_Training_Public/'
POST_EVENT_PREFIX = BASE_PREFIX + 'POST-event/'
ANNOT_PREFIX = BASE_PREFIX + 'annotations/'

print("Connexion S3 prête.")
print("Bucket :", BUCKET_NAME)
print("Post-event prefix :", POST_EVENT_PREFIX)
print("Annotations prefix :", ANNOT_PREFIX)

Connexion S3 prête.
Bucket : spacenet-dataset
Post-event prefix : spacenet/SN8_floods/Germany_Training_Public/POST-event/
Annotations prefix : spacenet/SN8_floods/Germany_Training_Public/annotations/


création des dossiers locaux

In [3]:
# Local folders
RAW_DATA_PATH = '../dattest/rawtest/'
IMAGES_PATH = os.path.join(RAW_DATA_PATH, 'images')
ANNOTATIONS_PATH = os.path.join(RAW_DATA_PATH, 'annotations')
PROCESSED_PATH = '../dattest/processedtest/'
MODELS_PATH = '../models/'

# Create folders if they do not exist
os.makedirs(IMAGES_PATH, exist_ok=True)
os.makedirs(ANNOTATIONS_PATH, exist_ok=True)
os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)

print("Dossiers créés avec succès.")
print("Images path      :", IMAGES_PATH)
print("Annotations path :", ANNOTATIONS_PATH)
print("Processed path   :", PROCESSED_PATH)
print("Models path      :", MODELS_PATH)

Dossiers créés avec succès.
Images path      : ../dattest/rawtest/images
Annotations path : ../dattest/rawtest/annotations
Processed path   : ../dattest/processedtest/
Models path      : ../models/


Fonction pour lister les fichiers du bucket

In [4]:
def list_s3_files(bucket_name, prefix):
    """
    List all files under a given S3 prefix.
    Handles pagination automatically.
    """
    files = []
    continuation_token = None

    while True:
        kwargs = {
            'Bucket': bucket_name,
            'Prefix': prefix,
            'MaxKeys': 1000
        }

        if continuation_token:
            kwargs['ContinuationToken'] = continuation_token

        response = s3.list_objects_v2(**kwargs)

        if 'Contents' in response:
            for obj in response['Contents']:
                key = obj['Key']
                if not key.endswith('/'):
                    files.append(key)

        if response.get('IsTruncated'):
            continuation_token = response.get('NextContinuationToken')
        else:
            break

    return files

Lister les images et annotations disponibles

In [5]:
# Get all post-event images and annotation files from S3
post_event_files = list_s3_files(BUCKET_NAME, POST_EVENT_PREFIX)
annot_files = list_s3_files(BUCKET_NAME, ANNOT_PREFIX)

print("Nombre d'images post-event trouvées :", len(post_event_files))
print("Nombre de fichiers annotations trouvés :", len(annot_files))

print("\n===== EXEMPLES DE NOMS D'IMAGES =====")
for f in post_event_files[:20]:
    print(os.path.basename(f))

print("\n===== EXEMPLES DE NOMS D'ANNOTATIONS =====")
for f in annot_files[:20]:
    print(os.path.basename(f))

Nombre d'images post-event trouvées : 282
Nombre de fichiers annotations trouvés : 202

===== EXEMPLES DE NOMS D'IMAGES =====
1040050035DC3B00_0_15_63.tif
1040050035DC3B00_0_15_65.tif
1040050035DC3B00_0_15_66.tif
1040050035DC3B00_0_15_67.tif
1040050035DC3B00_0_15_68.tif
1040050035DC3B00_0_15_69.tif
1040050035DC3B00_0_15_70.tif
1040050035DC3B00_0_16_63.tif
1040050035DC3B00_0_16_66.tif
1040050035DC3B00_0_16_67.tif
1040050035DC3B00_0_16_68.tif
1040050035DC3B00_0_16_70.tif
1040050035DC3B00_0_17_63.tif
1040050035DC3B00_0_17_66.tif
1040050035DC3B00_0_17_67.tif
1040050035DC3B00_0_17_68.tif
1040050035DC3B00_0_18_63.tif
1040050035DC3B00_0_18_65.tif
1040050035DC3B00_0_18_66.tif
1040050035DC3B00_0_18_67.tif

===== EXEMPLES DE NOMS D'ANNOTATIONS =====
0_15_63.geojson
0_15_65.geojson
0_15_66.geojson
0_15_67.geojson
0_15_68.geojson
0_15_69.geojson
0_15_70.geojson
0_16_63.geojson
0_16_66.geojson
0_16_67.geojson
0_16_68.geojson
0_16_70.geojson
0_17_63.geojson
0_17_66.geojson
0_17_67.geojson
0_17_68.ge

Fonction utilitaire pour enlever l’extension

In [6]:
def filename_without_extension(path):
    """
    Return the file name without extension.
    """
    return os.path.splitext(os.path.basename(path))[0]

Construire les paires par suffixe de tuile

In [7]:
matched_pairs = []

for annot_key in annot_files:
    # Annotation base name, e.g. 0_15_63
    annot_name = filename_without_extension(annot_key)

    found_img = None

    # Look for the image whose file name contains the annotation tile name
    for img_key in post_event_files:
        img_name = filename_without_extension(img_key)

        if annot_name in img_name:
            found_img = img_key
            break

    if found_img is not None:
        matched_pairs.append((found_img, annot_key))

# Remove duplicates if any
matched_pairs = list(dict.fromkeys(matched_pairs))

print("Nombre de paires trouvées :", len(matched_pairs))

print("\nExemples de paires :")
for pair in matched_pairs[:10]:
    print(pair)

Nombre de paires trouvées : 202

Exemples de paires :
('spacenet/SN8_floods/Germany_Training_Public/POST-event/1040050035DC3B00_0_15_63.tif', 'spacenet/SN8_floods/Germany_Training_Public/annotations/0_15_63.geojson')
('spacenet/SN8_floods/Germany_Training_Public/POST-event/1040050035DC3B00_0_15_65.tif', 'spacenet/SN8_floods/Germany_Training_Public/annotations/0_15_65.geojson')
('spacenet/SN8_floods/Germany_Training_Public/POST-event/1040050035DC3B00_0_15_66.tif', 'spacenet/SN8_floods/Germany_Training_Public/annotations/0_15_66.geojson')
('spacenet/SN8_floods/Germany_Training_Public/POST-event/1040050035DC3B00_0_15_67.tif', 'spacenet/SN8_floods/Germany_Training_Public/annotations/0_15_67.geojson')
('spacenet/SN8_floods/Germany_Training_Public/POST-event/1040050035DC3B00_0_15_68.tif', 'spacenet/SN8_floods/Germany_Training_Public/annotations/0_15_68.geojson')
('spacenet/SN8_floods/Germany_Training_Public/POST-event/1040050035DC3B00_0_15_69.tif', 'spacenet/SN8_floods/Germany_Training_Publi

Choisir combien de paires télécharger

In [8]:
# Limit number of pairs to download
MAX_FILES = 20
selected_pairs = matched_pairs[:MAX_FILES]

print("Nombre de paires sélectionnées :", len(selected_pairs))

Nombre de paires sélectionnées : 20


Fonction de téléchargement conditionnel

In [9]:
def download_if_missing(bucket_name, s3_key, local_path):
    """
    Download a file only if it does not already exist locally.
    """
    if not os.path.exists(local_path):
        s3.download_file(bucket_name, s3_key, local_path)
        print("Téléchargé :", local_path)
    else:
        print("Déjà présent :", local_path)

Télécharger les paires sélectionnées

In [10]:
for img_key, annot_key in selected_pairs:
    img_filename = os.path.basename(img_key)
    annot_filename = os.path.basename(annot_key)

    local_img_path = os.path.join(IMAGES_PATH, img_filename)
    local_annot_path = os.path.join(ANNOTATIONS_PATH, annot_filename)

    download_if_missing(BUCKET_NAME, img_key, local_img_path)
    download_if_missing(BUCKET_NAME, annot_key, local_annot_path)

print("Téléchargement terminé.")

Déjà présent : ../dattest/rawtest/images\1040050035DC3B00_0_15_63.tif
Déjà présent : ../dattest/rawtest/annotations\0_15_63.geojson
Déjà présent : ../dattest/rawtest/images\1040050035DC3B00_0_15_65.tif
Déjà présent : ../dattest/rawtest/annotations\0_15_65.geojson
Déjà présent : ../dattest/rawtest/images\1040050035DC3B00_0_15_66.tif
Déjà présent : ../dattest/rawtest/annotations\0_15_66.geojson
Déjà présent : ../dattest/rawtest/images\1040050035DC3B00_0_15_67.tif
Déjà présent : ../dattest/rawtest/annotations\0_15_67.geojson
Téléchargé : ../dattest/rawtest/images\1040050035DC3B00_0_15_68.tif
Téléchargé : ../dattest/rawtest/annotations\0_15_68.geojson
Téléchargé : ../dattest/rawtest/images\1040050035DC3B00_0_15_69.tif
Téléchargé : ../dattest/rawtest/annotations\0_15_69.geojson
Téléchargé : ../dattest/rawtest/images\1040050035DC3B00_0_15_70.tif
Téléchargé : ../dattest/rawtest/annotations\0_15_70.geojson
Téléchargé : ../dattest/rawtest/images\1040050035DC3B00_0_16_63.tif
Téléchargé : ../datt

un résumé local après téléchargement

In [11]:
local_images = sorted([
    f for f in os.listdir(IMAGES_PATH)
    if f.lower().endswith('.tif')
])

local_annots = sorted([
    f for f in os.listdir(ANNOTATIONS_PATH)
    if f.lower().endswith('.geojson')
])

print("Nombre d'images locales :", len(local_images))
print("Nombre d'annotations locales :", len(local_annots))

print("\nPremières images locales :")
print(local_images[:10])

print("\nPremières annotations locales :")
print(local_annots[:10])

Nombre d'images locales : 20
Nombre d'annotations locales : 20

Premières images locales :
['1040050035DC3B00_0_15_63.tif', '1040050035DC3B00_0_15_65.tif', '1040050035DC3B00_0_15_66.tif', '1040050035DC3B00_0_15_67.tif', '1040050035DC3B00_0_15_68.tif', '1040050035DC3B00_0_15_69.tif', '1040050035DC3B00_0_15_70.tif', '1040050035DC3B00_0_16_63.tif', '1040050035DC3B00_0_16_66.tif', '1040050035DC3B00_0_16_67.tif']

Premières annotations locales :
['0_15_63.geojson', '0_15_65.geojson', '0_15_66.geojson', '0_15_67.geojson', '0_15_68.geojson', '0_15_69.geojson', '0_15_70.geojson', '0_16_63.geojson', '0_16_66.geojson', '0_16_67.geojson']


une vérification locale des paires

In [12]:
def build_local_pairs_by_suffix(images_path, annotations_path):
    local_image_files = sorted([
        os.path.join(images_path, f)
        for f in os.listdir(images_path)
        if f.lower().endswith('.tif')
    ])

    local_annot_files = sorted([
        os.path.join(annotations_path, f)
        for f in os.listdir(annotations_path)
        if f.lower().endswith('.geojson')
    ])

    valid_pairs = []

    for annot_path in local_annot_files:
        annot_name = filename_without_extension(annot_path)

        found_img = None
        for img_path in local_image_files:
            img_name = filename_without_extension(img_path)

            if annot_name in img_name:
                found_img = img_path
                break

        if found_img is not None:
            valid_pairs.append((found_img, annot_path))

    valid_pairs = list(dict.fromkeys(valid_pairs))
    return valid_pairs

valid_local_pairs = build_local_pairs_by_suffix(IMAGES_PATH, ANNOTATIONS_PATH)

print("Nombre de paires locales valides :", len(valid_local_pairs))
for pair in valid_local_pairs[:10]:
    print(pair)

Nombre de paires locales valides : 20
('../dattest/rawtest/images\\1040050035DC3B00_0_15_63.tif', '../dattest/rawtest/annotations\\0_15_63.geojson')
('../dattest/rawtest/images\\1040050035DC3B00_0_15_65.tif', '../dattest/rawtest/annotations\\0_15_65.geojson')
('../dattest/rawtest/images\\1040050035DC3B00_0_15_66.tif', '../dattest/rawtest/annotations\\0_15_66.geojson')
('../dattest/rawtest/images\\1040050035DC3B00_0_15_67.tif', '../dattest/rawtest/annotations\\0_15_67.geojson')
('../dattest/rawtest/images\\1040050035DC3B00_0_15_68.tif', '../dattest/rawtest/annotations\\0_15_68.geojson')
('../dattest/rawtest/images\\1040050035DC3B00_0_15_69.tif', '../dattest/rawtest/annotations\\0_15_69.geojson')
('../dattest/rawtest/images\\1040050035DC3B00_0_15_70.tif', '../dattest/rawtest/annotations\\0_15_70.geojson')
('../dattest/rawtest/images\\1040050035DC3B00_0_16_63.tif', '../dattest/rawtest/annotations\\0_16_63.geojson')
('../dattest/rawtest/images\\1040050035DC3B00_0_16_66.tif', '../dattest/ra